# Ridge Regression - In Depth (with custom implementation)

This notebook explains **Ridge regression** from first principles and implements it **from scratch** with `numpy` (no `sklearn` model). By the end you should understand:

1. What the Ridge cost function is and how it differs from Ordinary Least Squares (OLS).
2. *Why* the L2 penalty helps (overfitting, bias-variance, multicollinearity, invertibility).
3. The closed-form solution and how to derive it with calculus.
4. The practical details (intercept handling, centering, numerical stability).
5. How the penalty strength `alpha` controls the model.


## 1. The model and the cost function

We still assume a **linear** relationship. For one sample with features $x$ and target $y$:

$$\hat{y} = x^{\top}\beta + b$$

For the whole dataset $\mathbf{X}$ (rows = samples, columns = features) and target $y$:

$$\hat{y} = \mathbf{X}\beta + b$$

**OLS** picks $\beta$ to minimize the *sum of squared residuals*:

$$J_{OLS}(\beta) = \|\mathbf{X}\beta + b - y\|^{2}$$

**Ridge** adds an **L2 penalty** - the squared magnitude of the weights - so large weights are discouraged:

$$J_{Ridge}(\beta) = \|\mathbf{X}\beta + b - y\|^{2} + \alpha\,\|\beta\|_{2}^{2}$$

where $\|\beta\|_{2}^{2} = \sum_j \beta_j^{2}$ and $\alpha \ge 0$ is the **regularization strength**. Larger $\alpha$ -> stronger shrinkage of the coefficients toward zero.


## 2. Why penalize the weights?

- **Overfitting / bias-variance:** OLS fits the training data perfectly when features are many or correlated, but the resulting weights can be large and cancel each other out - great on training data, poor on new data. The L2 penalty trades a little *bias* for much less *variance*.
- **Multicollinearity:** when features are highly correlated, $\mathbf{X}^{\top}\mathbf{X}$ is nearly singular, so OLS weights become unstable/huge. The penalty pushes the solution to a well-behaved one.
- **Invertibility:** the term $\alpha\mathbf{I}$ guarantees $\mathbf{X}^{\top}\mathbf{X} + \alpha\mathbf{I}$ is invertible even when $\mathbf{X}^{\top}\mathbf{X}$ is not.

Note the penalty is on $\beta$ (the feature weights) **only**, not on the intercept $b$ - we do not want to shrink the overall output level.


## 3. Closed-form solution (derivation)

To keep the math clean, absorb the intercept by **centering** the data: subtract each feature's mean and the target's mean. After centering, the model has no separate intercept and the cost is:

$$J(\beta) = (\mathbf{X}_c\beta - y_c)^{\top}(\mathbf{X}_c\beta - y_c) + \alpha\,\beta^{\top}\beta$$

Take the gradient w.r.t. $\beta$ (using $\frac{d}{d\beta}\beta^{\top}\beta = 2\beta$ and $\frac{d}{d\beta}(\mathbf{X}\beta)^{\top}(\mathbf{X}\beta) = 2\mathbf{X}^{\top}\mathbf{X}\beta$):

$$\nabla_\beta J = 2\mathbf{X}_c^{\top}(\mathbf{X}_c\beta - y_c) + 2\alpha\beta$$

Set the gradient to zero:

$$\mathbf{X}_c^{\top}\mathbf{X}_c\beta - \mathbf{X}_c^{\top}y_c + \alpha\beta = 0$$
$$(\mathbf{X}_c^{\top}\mathbf{X}_c + \alpha\mathbf{I})\beta = \mathbf{X}_c^{\top}y_c$$

So the closed-form (Normal Equation) solution is:

$$\boxed{\hat{\beta} = (\mathbf{X}_c^{\top}\mathbf{X}_c + \alpha\mathbf{I})^{-1}\,\mathbf{X}_c^{\top}y_c}$$

Once $\hat{\beta}$ is found, the intercept is recovered as $b = \bar{y} - \bar{x}^{\top}\hat{\beta}$.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Small dataset: size & rooms (correlated) + an irrelevant noise feature.
# This makes the L2 shrinkage visually clear.
np.random.seed(42)
n = 30
size = np.random.uniform(500, 3000, n)
rooms = size / 400 + np.random.randn(n) * 1.5
noise_feat = np.random.randn(n)
X = np.column_stack([size, rooms, noise_feat])
y = 100 * size + 50 * rooms + np.random.randn(n) * 2000
feature_names = ["size", "rooms", "noise_feat"]
print(pd.DataFrame(np.column_stack([X, y]), columns=feature_names + ["price"]).head())


## 4. Custom implementation

Step by step, matching the derivation above:

1. **Center** $\mathbf{X}$ and $y$ (subtract their means). Centering removes the intercept from the matrix math.
2. Build $\mathbf{A} = \mathbf{X}_c^{\top}\mathbf{X}_c + \alpha\mathbf{I}$. The $\alpha\mathbf{I}$ term is what makes Ridge special (and always invertible).
3. Solve $\mathbf{A}\beta = \mathbf{X}_c^{\top}y_c$ with `np.linalg.solve` (preferred over inverting $\mathbf{A}$ directly - it is numerically safer).
4. Recover the intercept: $b = \bar{y} - \bar{x}^{\top}\beta$.


In [ ]:
class RidgeRegression:
    """Ridge regression solved with the closed-form Normal Equation.

    beta = (Xc^T Xc + alpha * I)^-1 Xc^T yc
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        # 1. Center features and target (removes the intercept from the matrix).
        self.x_mean_ = X.mean(axis=0)
        Xc = X - self.x_mean_
        yc = y - y.mean()

        # 2. Build A = Xc^T Xc + alpha * I  (the L2 term lives in alpha*I).
        A = Xc.T @ Xc + self.alpha * np.eye(Xc.shape[1])

        # 3. Solve A * beta = Xc^T yc. (No need to invert A explicitly.)
        self.coef_ = np.linalg.solve(A, Xc.T @ yc)

        # 4. Recover the intercept on the original (uncentered) scale.
        self.intercept_ = y.mean() - self.x_mean_ @ self.coef_
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_


## 5. Fit and compare: OLS vs. Ridge

OLS is just Ridge with $\alpha = 0$. Notice how the **irrelevant `noise_feat`** weight shrinks under Ridge, while the meaningful `size`/`rooms` weights stay informative.


In [ ]:
ols = RidgeRegression(alpha=0.0).fit(X, y)     # equivalent to plain least squares
ridge = RidgeRegression(alpha=10.0).fit(X, y)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "OLS (a=0)": np.round(ols.coef_, 2),
    "Ridge (a=10)": np.round(ridge.coef_, 2),
})
print(coef_df.to_string(index=False))
print(f"\nIntercept  OLS   : {ols.intercept_:.2f}")
print(f"Intercept  Ridge : {ridge.intercept_:.2f}")

# Sanity check: does our custom code match sklearn? (sklearn optional)
try:
    from sklearn.linear_model import Ridge as SkRidge
    sk = SkRidge(alpha=10.0, fit_intercept=True).fit(X, y)
    print("\n[validate] max |coef diff vs sklearn|: "
          f"{np.max(np.abs(ridge.coef_ - sk.coef_)):.2e}")
except ImportError:
    pass


In [ ]:
x_pos = np.arange(len(feature_names))
width = 0.35

plt.bar(x_pos - width/2, ols.coef_, width, label="OLS (a=0)", color="gray")
plt.bar(x_pos + width/2, ridge.coef_, width, label="Ridge (a=10)", color="teal")
plt.xticks(x_pos, feature_names)
plt.ylabel("Coefficient")
plt.title("Ridge shrinks coefficients toward zero")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 6. Coefficient path vs. alpha (the Ridge signature)

As $\alpha$ increases, **every** coefficient is pulled smoothly toward zero - this is the defining behavior of L2 regularization (unlike Lasso, which can force coefficients exactly to 0).


In [ ]:
alphas = np.logspace(-2, 4, 25)
paths = np.array([RidgeRegression(alpha=a).fit(X, y).coef_ for a in alphas])

plt.figure(figsize=(7, 4.5))
for j, name in enumerate(feature_names):
    plt.plot(alphas, paths[:, j], marker="o", markersize=3, label=name)
plt.xscale("log")
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("alpha (log scale)")
plt.ylabel("Coefficient")
plt.title("Ridge coefficient path")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


## 7. Key takeaways

- **Formula:** $\hat{\beta} = (\mathbf{X}^{\top}\mathbf{X} + \alpha\mathbf{I})^{-1}\mathbf{X}^{\top}y$ (after centering).
- **L2 penalty** $\alpha\|\beta\|^{2}$ shrinks weights and improves generalization on noisy/correlated data.
- $\alpha = 0$ recovers OLS; $\alpha \to \infty$ drives all coefficients to 0.
- The $\alpha\mathbf{I}$ term guarantees a unique, stable solution even when features are collinear.
- The intercept is **not** penalized; we handle it by centering, then add it back.
